In [ ]:
import os
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [ ]:
class ChestXRayDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        self.label_map = {
            "normal": 0,
            "bacterial": 1,
            "viral": 2
        }

        for label_name in os.listdir(root_dir):
            full_path = os.path.join(root_dir, label_name)
            if os.path.isdir(full_path) and label_name in self.label_map:
                for file in os.listdir(full_path):
                    self.samples.append((os.path.join(full_path, file), self.label_map[label_name]))
            else:
                print(f"Skipping unknown or invalid class: {label_name}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, label = self.samples[idx]
        try:
            image = Image.open(image_path).convert("RGB")
        except:
            print(f"Error loading image: {image_path}")
            image = Image.new("RGB", (224, 224))

        if self.transform:
            image = self.transform(image)
        return image, label


In [ ]:
image_size = 224
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_dataset = ChestXRayDataset("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/trained", transform)
val_dataset   = ChestXRayDataset("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/valed", transform)
test_dataset  = ChestXRayDataset("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/tested", transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=16, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=16, num_workers=2)

In [ ]:
def count_images_per_class(dataset_path, dataset_name):
    print(f"\n{dataset_name} Dataset Summary:")
    label_map = {"normal": 0, "bacterial": 1, "viral": 2}
    total = 0
    for class_name in label_map.keys():
        class_path = os.path.join(dataset_path, class_name)
        if os.path.exists(class_path):
            count = len([f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
            print(f" --  {class_name.capitalize():9}: {count} images")
            total += count
        else:
            print(f"  {class_name} class folder missing!")
    print(f"  Total images: {total}\n")

# Run counts for all datasets
count_images_per_class("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/trained", "Train")
count_images_per_class("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/valed", "Validation")
count_images_per_class("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/tested", "Test")



Train Dataset Summary:
 --  Normal   : 2524 images
 --  Bacterial: 2529 images
 --  Viral    : 2508 images
  Total images: 7561


Validation Dataset Summary:
 --  Normal   : 100 images
 --  Bacterial: 100 images
 --  Viral    : 100 images
  Total images: 300


Test Dataset Summary:
 --  Normal   : 303 images
 --  Bacterial: 302 images
 --  Viral    : 304 images
  Total images: 909



In [ ]:
class ResNetClassifier(nn.Module):
    def __init__(self, num_classes=3):
        super(ResNetClassifier, self).__init__()
        base_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.features = nn.Sequential(*list(base_model.children())[:-1])  # Remove final FC
        self.classifier = nn.Linear(2048, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten
        return self.classifier(x)

In [ ]:
def train_model(model, epochs=5):
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0

        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            predicted = torch.argmax(outputs, dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            if batch_idx % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Step [{batch_idx}], Loss: {loss.item():.4f}")

        acc = 100 * correct / total
        val_acc = validate_model(model)
        print(f"Epoch {epoch+1}/{epochs} - Train Acc: {acc:.2f}%, Val Acc: {val_acc:.2f}%")

In [ ]:
def validate_model(model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            predicted = torch.argmax(outputs, dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    model.train()
    return 100 * correct / total

In [ ]:
def evaluate_model(model):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    print("\nConfusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=["normal", "bacterial", "viral"]))

In [ ]:
model = ResNetClassifier().to(device)
train_model(model, epochs=5)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 135MB/s]


Epoch [1/5], Step [0], Loss: 1.1027
Epoch [1/5], Step [10], Loss: 0.8564
Epoch [1/5], Step [20], Loss: 0.6305
Epoch [1/5], Step [30], Loss: 0.5369
Epoch [1/5], Step [40], Loss: 0.7411
Epoch [1/5], Step [50], Loss: 0.4410
Epoch [1/5], Step [60], Loss: 0.6601
Epoch [1/5], Step [70], Loss: 0.7426
Epoch [1/5], Step [80], Loss: 0.8023
Epoch [1/5], Step [90], Loss: 0.4751
Epoch [1/5], Step [100], Loss: 0.3256
Epoch [1/5], Step [110], Loss: 0.4591
Epoch [1/5], Step [120], Loss: 0.3118
Epoch [1/5], Step [130], Loss: 0.2847
Epoch [1/5], Step [140], Loss: 0.2857
Epoch [1/5], Step [150], Loss: 0.5470
Epoch [1/5], Step [160], Loss: 0.4469
Epoch [1/5], Step [170], Loss: 0.2132
Epoch [1/5], Step [180], Loss: 0.5561
Epoch [1/5], Step [190], Loss: 0.1809
Epoch [1/5], Step [200], Loss: 0.5312
Epoch [1/5], Step [210], Loss: 0.3297
Epoch [1/5], Step [220], Loss: 0.4578
Epoch [1/5], Step [230], Loss: 0.3403
Epoch [1/5], Step [240], Loss: 0.2595
Epoch [1/5], Step [250], Loss: 0.6095
Epoch [1/5], Step [260]

In [ ]:

print("\nTesting Accuracy:")
evaluate_model(model)


Testing Accuracy:

Confusion Matrix:
[[242   6  55]
 [  4 269  29]
 [  2  15 287]]

Classification Report:
              precision    recall  f1-score   support

      normal       0.98      0.80      0.88       303
   bacterial       0.93      0.89      0.91       302
       viral       0.77      0.94      0.85       304

    accuracy                           0.88       909
   macro avg       0.89      0.88      0.88       909
weighted avg       0.89      0.88      0.88       909

